# Construction des features – Prix des denrées sur les marchés du Bénin

Ce notebook construit le fichier `data/features.parquet`, prêt pour la modélisation. La liste des features vient de la partie 4 de `02_eda.ipynb` : seules les pistes confirmées par un chiffre sont gardées. Le notebook ne fait pas d'analyse : il appelle la fonction de `src/features.py`, vérifie le résultat et enregistre le fichier.

Le notebook suit quatre étapes :
1. Construction
2. Contrôles
3. Enregistrement
4. Description des features

## Convention à retenir

Une ligne du fichier = **une série (un produit sur un marché) et un mois M à prévoir**.
- La cible est `prix` : le prix du mois M.
- Chaque feature n'utilise que des mois **strictement avant M**. Aucune feature ne dépend d'un prix du mois M ou d'après.
- Une feature vide veut dire « on ne la connaît pas » (série trop récente, mois sans relevé). On ne la remplit jamais en copiant une valeur du futur.

## 5.1 Construction

La fonction `construire_features` (dans `src/features.py`) remet les séries sur un calendrier mensuel complet, puis calcule chaque feature en décalant les données. On charge le fichier propre et on appelle la fonction.

In [1]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np

from src.features import construire_features, verifier_sans_fuite, FEATURES

prix = pd.read_parquet("../data/prix_propres.parquet")
feats = construire_features(prix)

print(prix.shape, "->", feats.shape)
feats.head()

(21394, 13) -> (21394, 18)


,commodity,produit,market,admin1,mois,date,annee,prix,prix_m1,prix_m12,prix_dernier_connu,anciennete_dernier_prix,med_nat_m1,var_nat_m1,ecart_marche,mois_annee,latitude,nord
0,Beans (white),Haricot blanc,Abomey,Zou,2019-07,2019-07-01,2019,434.78,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7,7.18,0
1,Beans (white),Haricot blanc,Abomey,Zou,2019-08,2019-08-01,2019,434.78,434.78,NaN,434.78,1.0,485.060,NaN,NaN,8,7.18,0
2,Beans (white),Haricot blanc,Abomey,Zou,2019-09,2019-09-01,2019,434.78,434.78,NaN,434.78,1.0,441.575,0.0,NaN,9,7.18,0
3,Beans (white),Haricot blanc,Abomey,Zou,2019-10,2019-10-01,2019,434.78,434.78,NaN,434.78,1.0,434.780,0.0,NaN,10,7.18,0
4,Beans (white),Haricot blanc,Abomey,Zou,2020-02,2020-02-01,2020,434.78,NaN,NaN,NaN,NaN,496.180,0.0,NaN,2,7.18,0


**Conclusion.** On a une ligne par prix relevé : le nombre de lignes est celui de `prix_propres.parquet`, chaque relevé devenant une cible à prévoir. Les colonnes sont l'identifiant de la ligne (produit, marché, mois), la cible `prix`, et les features de la liste de 4.3.

## 5.2 Contrôles

### Une seule ligne par série et par mois, et aucun relevé perdu

In [2]:
print("lignes en double :", feats.duplicated(["commodity", "market", "mois"]).sum())
print("lignes dans le fichier propre :", len(prix), "| lignes dans les features :", len(feats))
print("séries :", feats.groupby(["commodity", "market"]).ngroups)
print("période :", feats["mois"].min(), "->", feats["mois"].max())

lignes en double : 0
lignes dans le fichier propre : 21394 | lignes dans les features : 21394
séries : 313
période : 2019-01 -> 2026-02


### Les décalages sont bien les prix des mois d'avant

On compare `prix_m1` et `prix_m12` au prix brut du fichier propre, retrouvé à la main avec la date décalée. Les deux doivent être identiques partout où la feature n'est pas vide.

In [3]:
brut = prix.assign(mois=prix["date"].dt.to_period("M"))[["commodity", "market", "mois", "price"]]

for nb_mois, colonne in [(1, "prix_m1"), (12, "prix_m12")]:
    decale = brut.assign(mois=brut["mois"] + nb_mois).rename(columns={"price": "attendu"})
    t = feats.merge(decale, on=["commodity", "market", "mois"], how="left")
    connus = t[colonne].notna()
    print(colonne, ": écarts avec le prix brut =", (t.loc[connus, colonne] != t.loc[connus, "attendu"]).sum(),
          "| prix vides alors que le brut existe =", (t[colonne].isna() & t["attendu"].notna()).sum())

prix_m1 : écarts avec le prix brut = 0 | prix vides alors que le brut existe = 0
prix_m12 : écarts avec le prix brut = 0 | prix vides alors que le brut existe = 0


### Aucune feature ne voit le futur

Pour trois mois M, on multiplie par 7 tous les prix des mois M et d'après, on reconstruit les features, et on compare les lignes du mois M. Si une feature utilisait le mois M ou un mois plus tard, elle changerait. Le résultat est le nombre de colonnes qui changent : on attend 0.

In [4]:
verifier_sans_fuite(prix, feats, mois_test=("2021-06", "2023-03", "2025-01"))

{'2021-06': 0, '2023-03': 0, '2025-01': 0}

**Conclusion.** Aucune ligne en double, aucun relevé perdu (21 394 lignes, 313 séries, de janvier 2019 à février 2026). `prix_m1` et `prix_m12` sont identiques aux prix bruts décalés, et aucune feature ne change quand on modifie les prix du mois M et des mois d'après : il n'y a pas de fuite du futur.

### Valeurs manquantes

On compte, pour chaque feature, la part de lignes vides, puis par produit. Les trous de la partie 1 (début 2019, mars 2022, séries qui commencent tard) se retrouvent ici.

In [5]:
manquants = feats[FEATURES].isna().mean().mul(100).round(1).to_frame("lignes vides (%)")
manquants["lignes vides"] = feats[FEATURES].isna().sum()
manquants

,lignes vides (%),lignes vides
prix_m1,15.5,3320
prix_m12,26.3,5634
prix_dernier_connu,2.8,607
anciennete_dernier_prix,2.8,607
med_nat_m1,3.2,679
var_nat_m1,7.2,1534
ecart_marche,8.9,1913
mois_annee,0.0,0
latitude,0.0,0
nord,0.0,0


In [6]:
(feats.groupby("produit")[FEATURES].apply(lambda g: g.isna().mean() * 100)
 .round(0).loc[["Gari", "Riz importé", "Maïs blanc", "Haricot blanc", "Tomate", "Huile d'arachide", "Oignon"],
               ["prix_m1", "prix_m12", "prix_dernier_connu", "med_nat_m1", "var_nat_m1", "ecart_marche"]])

,prix_m1,prix_m12,prix_dernier_connu,med_nat_m1,var_nat_m1,ecart_marche
produit,,,,,,
Gari,14.0,25.0,3.0,3.0,7.0,9.0
Riz importé,17.0,28.0,3.0,3.0,7.0,9.0
Maïs blanc,17.0,28.0,3.0,4.0,8.0,10.0
Haricot blanc,15.0,25.0,3.0,3.0,7.0,9.0
Tomate,14.0,25.0,3.0,3.0,8.0,9.0
Huile d'arachide,15.0,26.0,3.0,3.0,7.0,9.0
Oignon,16.0,27.0,3.0,3.0,7.0,9.0


In [7]:
par_annee = feats.groupby("annee")[["prix_m1", "prix_m12", "var_nat_m1", "ecart_marche"]].apply(lambda g: g.isna().mean() * 100).round(0)
par_annee["lignes"] = feats.groupby("annee").size()
par_annee

,prix_m1,prix_m12,var_nat_m1,ecart_marche,lignes
annee,,,,,
2019,36.0,100.0,49.0,87.0,1682
2020,17.0,53.0,6.0,15.0,3088
2021,11.0,17.0,0.0,0.0,3287
2022,20.0,11.0,18.0,0.0,2991
2023,12.0,19.0,0.0,0.0,3303
2024,12.0,11.0,0.0,0.0,3229
2025,11.0,13.0,0.0,0.0,3296
2026,15.0,10.0,0.0,0.0,518


**Conclusion.** Le prix du mois d'avant manque pour 15,5 % des lignes (3 320) et celui d'il y a 12 mois pour 26,3 % (5 634). Avec le dernier prix connu (3 mois au plus), il ne reste que 2,8 % de lignes sans repère (607). Les trous se concentrent au début : en 2019, `prix_m12` est vide à 100 %, `prix_m1` à 36 % et `ecart_marche` à 87 % (il faut 6 relevés passés) ; en 2020, `prix_m12` est vide à 53 %. En 2022, mars sans relevé laisse `prix_m1` vide à 20 % et `var_nat_m1` à 18 %. Les produits sont à égalité (`prix_m1` vide de 14 à 17 % selon le produit). À partir de 2021, `ecart_marche` est toujours rempli, et `var_nat_m1` aussi sauf en 2022.

### Les features sont-elles reliées à la cible ?

Ce n'est pas un test de leur utilité (la modélisation le fera), seulement un contrôle de bon sens : les features que 2.5 annonçait comme très liées au prix doivent l'être ici. On calcule la corrélation de chaque feature avec `prix`, produit par produit.

In [8]:
colonnes_prix = ["prix_m1", "prix_m12", "prix_dernier_connu", "med_nat_m1", "latitude"]
ordre = ["Gari", "Riz importé", "Maïs blanc", "Haricot blanc", "Tomate", "Huile d'arachide", "Oignon"]

lien = pd.DataFrame({c: feats.groupby("produit").apply(lambda g: g["prix"].corr(g[c])) for c in colonnes_prix})
lien["ecart_marche"] = feats.groupby("produit").apply(lambda g: g["prix"].corr(g["ecart_marche"]))
lien.loc[ordre].round(2)

C:\Users\User\AppData\Local\Temp\ipykernel_58612\1037865105.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  lien = pd.DataFrame({c: feats.groupby("produit").apply(lambda g: g["prix"].corr(g[c])) for c in colonnes_prix})
C:\Users\User\AppData\Local\Temp\ipykernel_58612\1037865105.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  lien = pd.DataFrame({c: feats.groupby("produit").apply(lambda g: g["prix"].corr(g[c]

,prix_m1,prix_m12,prix_dernier_connu,med_nat_m1,latitude,ecart_marche
produit,,,,,,
Gari,0.93,0.48,0.92,0.49,0.31,0.63
Riz importé,0.93,0.55,0.92,0.56,-0.07,0.50
Maïs blanc,0.87,0.35,0.85,0.63,-0.20,0.46
Haricot blanc,0.89,0.46,0.88,0.54,-0.12,0.63
Tomate,0.72,0.51,0.68,0.45,-0.00,0.31
Huile d'arachide,0.94,0.46,0.93,0.74,0.24,0.46
Oignon,0.76,0.47,0.75,0.36,0.01,0.43


**Conclusion.** Les corrélations de `prix_m1` sont celles de 2.5 (0,93 pour le gari et le riz, 0,94 pour l'huile, 0,87 pour le maïs, 0,89 pour le haricot, 0,72 pour la tomate, 0,76 pour l'oignon), et celles de `prix_m12` aussi (0,35 à 0,55). `prix_dernier_connu` reste presque aussi bon que `prix_m1` (0,04 de moins au plus). `med_nat_m1` va de 0,36 (oignon) à 0,73 (huile). `ecart_marche` est le plus lié au prix pour le gari (0,68) et le haricot (0,65), moins pour la tomate (0,33). La latitude a le même signe qu'en 2.3 (gari +0,31, huile +0,24, maïs -0,20), en plus faible parce qu'on compte ici tous les relevés et non la médiane de chaque marché ; elle ne dit rien pour la tomate (0,00), l'oignon (0,01) et le riz (-0,07).

## 5.3 Enregistrement

On enregistre une ligne par série et par mois, avec la cible et les features. Le mois est écrit en texte (`2024-06`) parce que le format parquet ne gère pas le type « période » de pandas. Pour retrouver une période : `pd.PeriodIndex(feats["mois"], freq="M")`.

In [9]:
sortie = feats.assign(mois=feats["mois"].astype(str))
sortie.to_parquet("../data/features.parquet", index=False)

relu = pd.read_parquet("../data/features.parquet")
print(relu.shape)
relu.dtypes

(21394, 18)


commodity                          object
produit                            object
market                             object
admin1                             object
mois                               object
date                       datetime64[ns]
annee                               int64
prix                              float64
prix_m1                           float64
prix_m12                          float64
prix_dernier_connu                float64
anciennete_dernier_prix           float64
med_nat_m1                        float64
var_nat_m1                        float64
ecart_marche                      float64
mois_annee                          int64
latitude                          float64
nord                                int64
dtype: object

## 5.4 Description des features

Pour la personne qui modélise ensuite. Dans chaque cas, M est le mois à prévoir, et aucune feature n'utilise M ni un mois d'après.

**Colonnes d'identification (ne sont pas des features)**

| Colonne | Contenu |
|---|---|
| `commodity`, `produit` | Produit (nom d'origine et nom français) |
| `market`, `admin1` | Marché et département |
| `mois`, `date`, `annee` | Mois M à prévoir (texte `2024-06`, date, année) |
| `prix` | **La cible** : prix du mois M, en FCFA par kg (par litre pour l'huile) |

**Features**

| Feature | Définition | Mois utilisés |
|---|---|---|
| `prix_m1` | Prix de la série au mois M-1 | M-1 |
| `prix_m12` | Prix de la série au mois M-12 | M-12 |
| `prix_dernier_connu` | Dernier prix connu de la série, dans les 3 mois avant M | de M-3 à M-1 |
| `anciennete_dernier_prix` | Âge en mois de ce dernier prix (1 = c'est `prix_m1`) | de M-3 à M-1 |
| `med_nat_m1` | Médiane nationale du produit au mois M-1 (vide si le produit a moins de 10 séries ce mois-là) | M-1 |
| `var_nat_m1` | Variation en % de la médiane nationale entre M-2 et M-1 (vide si moins de 10 séries du produit) | M-2 et M-1 |
| `ecart_marche` | Moyenne des rapports « prix du marché / médiane nationale du mois » sur les mois passés (au moins 6 relevés) | tous les mois avant M |
| `mois_annee` | Mois de l'année de M (1 à 12) | M (le calendrier est connu d'avance) |
| `latitude` | Latitude du marché | fixe |
| `nord` | 1 si le marché est en Alibori, Atakora, Borgou ou Donga, 0 sinon | fixe |

**À savoir pour la modélisation**

- **Valeurs vides.** Elles sont volontaires et il y en a (voir 5.2). Les arbres de décision en boosting (LightGBM, `HistGradientBoostingRegressor`) les acceptent telles quelles ; pour les autres modèles, il faudra les traiter, sans jamais utiliser de valeur du futur.
- **Découpage.** On découpe dans le temps (par exemple entraînement jusqu'à fin 2024, test après), jamais au hasard. Les features étant décalées, le découpage dans le temps suffit pour éviter la fuite, mais l'écart habituel d'un marché et les médianes nationales sont calculés sur toute la période : en production, on les recalcule avec les données disponibles à la date de la prévision (c'est ce que fait la fonction, mois par mois).
- **2019.** Début 2019 compte peu de séries et `prix_m12` n'existe pas avant 2020. On peut commencer l'entraînement en 2020.
- **Tomate et oignon.** Le prix varie de 20 à 28 % par mois en moyenne (1.2) : on travaillera sur le log du prix (1.1), et le modèle « le prix du mois d'avant » (`prix_m1`) est la référence à battre.
- **Autres pistes.** Les marchés voisins, Dantokpa, Malanville et les décalages de 2 à 10 mois ont été écartés en 4.3, faute de chiffre qui les justifie.